
# NBA Advanced Stats (2000–2009): Real-World Data Storytelling

This notebook completes the assignment requirements for the **Real-World Data Storytelling** project using the Kaggle dataset **NBA Advanced Stats 2000–2009**.

## Story Focus
We want to understand how player efficiency, usage, and value changed across the decade, and whether some positions or teams consistently produced stronger advanced-stat profiles.

## Research Questions
1. How did average advanced metrics change across seasons from 2000 to 2009?
2. Which positions had the highest efficiency and value?
3. Is higher usage associated with better overall outcomes such as win shares or BPM?
4. Which teams and seasons stood out when we look only at players with meaningful minutes?
5. Does the distribution of efficiency suggest a balanced league or a small group of elite outliers?



## 1. Imports and Setup
We use pandas, NumPy, matplotlib, and seaborn as requested in the assignment.


In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
FIG_DIR = "figures"

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# Update this filename if needed to match the Kaggle CSV you downloaded.
DATA_PATH = os.path.join(RAW_DIR, "nba_advanced_stats_2000_2009.csv")
print("Looking for dataset at:", DATA_PATH)



## 2. Demonstrating `Series` and `DataFrame` by Hand
The assignment requires at least one small `pd.Series` and one small `pd.DataFrame` created manually.


In [ ]:

example_series = pd.Series({
    "PER_threshold": 15,
    "Elite_PER_threshold": 20,
    "Meaningful_minutes": 500
})

example_dataframe = pd.DataFrame({
    "metric": ["PER", "TS%", "USG%"],
    "meaning": ["efficiency", "shooting efficiency", "usage rate"]
})

display(example_series)
display(example_dataframe)



## 3. Data Loading and Initial Inspection
We load the CSV with `pd.read_csv(...)`, which satisfies the reading-data requirement.


In [ ]:

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.info())



### Initial Notes
This dataset is expected to contain advanced NBA player statistics for seasons from 2000 through 2009. Typical columns include identifiers like player name, team, and position, plus advanced metrics such as `PER`, `TS%`, `USG%`, `WS`, `BPM`, and `VORP`.



## 4. Cleaning and Transformation
Here we:
- standardize column names
- remove an index-like unnamed column if present
- convert numeric columns using `pd.to_numeric(..., errors="coerce")`
- convert a date-style column using `pd.to_datetime(...)`
- handle missing values using **two different strategies**
- create new derived columns using `.apply(...)` and `np.where(...)`


In [ ]:

df.columns = [c.strip() for c in df.columns]

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

if "Year" in df.columns:
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
    df = df[df["Year"].between(2000, 2009, inclusive="both")].copy()

numeric_candidates = [
    "Age", "G", "GS", "MP", "PER", "TS%", "3PAr", "FTr", "ORB%", "DRB%", "TRB%",
    "AST%", "STL%", "BLK%", "TOV%", "USG%", "OWS", "DWS", "WS", "WS/48",
    "OBPM", "DBPM", "BPM", "VORP", "FG", "FGA", "FG%", "3P", "3PA", "3P%",
    "2P", "2PA", "2P%", "eFG%", "FT", "FTA", "FT%", "ORB", "DRB", "TRB",
    "AST", "STL", "BLK", "TOV", "PF", "PTS"
]

numeric_cols = [c for c in numeric_candidates if c in df.columns]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Optional dtype cleanup for categorical columns
for c in ["Pos", "Tm"]:
    if c in df.columns:
        df[c] = df[c].astype("category")

# Date conversion requirement:
# The source may not include a native date column, so we create a season-end date from Year.
df["season_end_date"] = pd.to_datetime(
    df["Year"].astype("Int64").astype(str) + "-06-30",
    errors="coerce"
)

# Missing value strategy 1: drop rows missing key identifiers
critical_cols = [c for c in ["Player", "Pos", "Tm", "Year"] if c in df.columns]
df = df.dropna(subset=critical_cols)

# Missing value strategy 2: fill important numeric columns with the median
for c in [col for col in ["PER", "TS%", "USG%", "WS", "BPM", "VORP", "MP"] if col in df.columns]:
    df[c] = df[c].fillna(df[c].median())

# Derived columns
if "MP" in df.columns:
    df["minutes_bucket"] = np.where(
        df["MP"] >= 2000, "heavy_minutes",
        np.where(df["MP"] >= 1000, "rotation", "limited")
    )

if "PER" in df.columns:
    df["efficiency_tier"] = df["PER"].apply(
        lambda x: "elite" if x >= 20 else ("above_average" if x >= 15 else "below_average")
    )

display(df.head())
print(df[["Year", "season_end_date"]].head())



## 5. Indexing, Selection, and Filtering
The assignment asks for `.loc`, `.iloc`, and boolean filtering examples.


In [ ]:

# .loc example
loc_example = df.loc[
    df["Pos"].astype(str).str.contains("G", na=False),
    ["Player", "Year", "Pos", "Tm", "PER", "USG%"]
].head(10)

# .iloc example
iloc_example = df.iloc[:10, :6]

# Boolean filtering example
high_minute_high_per = df[(df["MP"] >= 1000) & (df["PER"] >= 15)]

display(loc_example)
display(iloc_example)
print("High-minute / high-PER rows:", high_minute_high_per.shape[0])



## 6. Descriptive Statistics
We use `.describe()` and explicit statistics such as mean, median, min, max, and standard deviation.


In [ ]:

key_metrics = [c for c in ["PER", "TS%", "USG%", "WS", "BPM", "VORP", "MP"] if c in df.columns]

display(df[key_metrics].describe().round(3))

explicit_stats = pd.DataFrame({
    "mean": df[key_metrics].mean(),
    "median": df[key_metrics].median(),
    "min": df[key_metrics].min(),
    "max": df[key_metrics].max(),
    "std": df[key_metrics].std()
}).round(3)

display(explicit_stats)



### Grouped Descriptive Statistics
We also need descriptive statistics by group for at least one categorical variable.


In [ ]:

position_group_stats = (
    df.groupby("Pos")[["PER", "TS%", "USG%", "WS", "BPM", "VORP"]]
      .agg(["count", "mean", "median"])
      .round(3)
)

display(position_group_stats)



## 7. Grouping and Aggregation
This section satisfies the requirement for `groupby(...).agg(...)`, including a **multi-column aggregation**.


In [ ]:

team_year_summary = (
    df[df["MP"] >= 500]
    .groupby(["Tm", "Year"])
    .agg(
        player_count=("Player", "count"),
        mean_PER=("PER", "mean"),
        mean_TS=("TS%", "mean"),
        mean_USG=("USG%", "mean"),
        mean_WS=("WS", "mean"),
        mean_BPM=("BPM", "mean")
    )
    .reset_index()
    .sort_values(["Year", "mean_PER"], ascending=[True, False])
)

display(team_year_summary.head(15))



## 8. Reshaping with a Pivot Table
This satisfies the simple reshaping requirement.


In [ ]:

position_per_pivot = df.pivot_table(
    index="Pos",
    columns="Year",
    values="PER",
    aggfunc="mean"
).round(2)

display(position_per_pivot)



## 9. Exporting Cleaned and Summary Data
Saving intermediate outputs makes the repository more professional and easier to review.


In [ ]:

cleaned_output = os.path.join(PROCESSED_DIR, "nba_advanced_stats_2000_2009_cleaned.csv")
league_output = os.path.join(PROCESSED_DIR, "league_avg_by_year.csv")
team_output = os.path.join(PROCESSED_DIR, "team_year_summary.csv")
pivot_output = os.path.join(PROCESSED_DIR, "position_per_by_year_pivot.csv")

df.to_csv(cleaned_output, index=False)

league_avg_by_year = (
    df.groupby("Year")[["PER", "TS%", "USG%", "WS", "BPM", "VORP"]]
      .mean()
      .round(3)
)
league_avg_by_year.to_csv(league_output)

team_year_summary.to_csv(team_output, index=False)
position_per_pivot.to_csv(pivot_output)

print("Saved:")
print(cleaned_output)
print(league_output)
print(team_output)
print(pivot_output)



## 10. Visualizations
The assignment requires at least:
- 1 line plot
- 1 bar plot
- 1 histogram
- 1 scatter plot
- clear labels and styling
- at least 3 figures saved to the `figures/` directory at `dpi=300`



### Line Plot: League Average Metrics Over Time


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))
league_avg_by_year[["PER", "TS%", "USG%"]].plot(ax=ax, marker="o")
ax.set_title("League Average Advanced Metrics by Season")
ax.set_xlabel("Season End Year")
ax.set_ylabel("Average Metric Value")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "line_league_metrics_over_time.png"), dpi=300)
plt.show()



This plot helps us see whether the league's player profile changed over the decade. If some lines trend upward, that suggests league-wide changes in style, efficiency, or player usage.



### Bar Plot: Average PER by Position


In [ ]:

fig, ax = plt.subplots(figsize=(9, 5))
pos_order = (
    df[df["MP"] >= 500]
    .groupby("Pos")["PER"]
    .mean()
    .sort_values(ascending=False)
    .index
)

sns.barplot(
    data=df[df["MP"] >= 500],
    x="Pos",
    y="PER",
    order=pos_order,
    estimator="mean",
    errorbar=None,
    ax=ax
)
ax.set_title("Average PER by Position (Players with at Least 500 Minutes)")
ax.set_xlabel("Position")
ax.set_ylabel("Average PER")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "bar_average_per_by_position.png"), dpi=300)
plt.show()



This bar chart compares positions directly. Restricting to players with at least 500 minutes removes many tiny-sample seasons that can distort averages.



### Histogram: Distribution of PER


In [ ]:

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df["PER"].dropna(), bins=30)
ax.set_title("Distribution of Player Efficiency Rating (PER)")
ax.set_xlabel("PER")
ax.set_ylabel("Frequency")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "hist_per_distribution.png"), dpi=300)
plt.show()



The histogram shows whether most players cluster near average efficiency or whether the data has a long tail of elite performers.



### Scatter Plot: Usage Rate vs Win Shares


In [ ]:

fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(
    data=df[df["MP"] >= 500],
    x="USG%",
    y="WS",
    hue="Pos",
    alpha=0.7,
    ax=ax
)
ax.set_title("Usage Rate vs Win Shares")
ax.set_xlabel("USG%")
ax.set_ylabel("WS")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "scatter_usage_vs_win_shares.png"), dpi=300)
plt.show()



This scatter plot helps answer whether heavier offensive involvement tends to align with greater overall player contribution.



## 11. Correlation Analysis
The assignment requires a correlation matrix and a visual summary.


In [ ]:

corr_cols = [c for c in ["PER", "TS%", "USG%", "WS", "BPM", "VORP", "AST%", "TRB%", "TOV%"] if c in df.columns]
corr_matrix = df[corr_cols].corr().round(3)

display(corr_matrix)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", ax=ax)
ax.set_title("Correlation Heatmap of Key Advanced Metrics")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "heatmap_correlation_key_metrics.png"), dpi=300)
plt.show()



### Interpreting the Correlation Heatmap
- Strong positive correlations suggest metrics that often increase together.
- Negative correlations can reveal tradeoffs, such as higher turnovers for some high-usage players.
- Correlation does not prove causation, but it helps identify relationships worth discussing.



## 12. Feature Exploration Questions
Below are two example stakeholder-friendly questions answered using grouped statistics and plots.



### Question 1: Which positions were most efficient on average?


In [ ]:

position_rank = (
    df[df["MP"] >= 500]
    .groupby("Pos")[["PER", "TS%", "WS", "BPM", "VORP"]]
    .mean()
    .sort_values("PER", ascending=False)
    .round(3)
)

display(position_rank)



**Plain-English interpretation:**  
The table above ranks positions by average performance on several advanced metrics. Positions near the top of the table can be described as more efficient or more valuable on average during the 2000–2009 period, especially once we remove low-minute seasons.



### Question 2: Which team-seasons stood out for player efficiency?


In [ ]:

best_team_seasons = (
    team_year_summary.sort_values(["mean_PER", "mean_WS"], ascending=False)
    .head(15)
)

display(best_team_seasons)



**Plain-English interpretation:**  
This summary highlights team-seasons where the rotation-level players posted especially strong efficiency profiles. It is useful for connecting player-level advanced stats back to team context.



## 13. Summary and Key Findings
You should update this section after running the notebook on the actual Kaggle CSV, but the likely structure is:

1. League-average advanced metrics either stayed relatively stable or shifted gradually across the decade.
2. Some positions likely show higher average efficiency and value than others, especially when small-minute players are excluded.
3. Usage alone does not guarantee value, but it may be positively related to metrics such as Win Shares and BPM for many regular contributors.
4. The distribution of PER is usually centered near league-average values, with a smaller number of high-end outliers.
5. Team context matters: certain team-seasons may stand out because they had several efficient rotation players at once.

When submitting, replace these general statements with observations based on your exact notebook outputs.
